# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [31]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix, f1_score


from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, precision_score, recall_score, f1_score

In [2]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [3]:
fraud.fraud.value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [ ]:
import seaborn as sns
sns.countplot(fraud.fraud)

In [4]:
features = fraud.drop(columns =['fraud'])
target = fraud.fraud
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [18]:
normalizer = MinMaxScaler()

normalizer.fit(X_train)
X_train_norm_np = normalizer.transform(X_train)

X_test_norm_np = normalizer.transform(X_test)
X_train_norm_df = pd.DataFrame(X_train_norm_np, columns = X_train.columns, index=X_train.index)
X_test_norm_df = pd.DataFrame(X_test_norm_np, columns = X_test.columns, index=X_test.index)

In [19]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled_np = scaler.transform(X_train)
X_test_scaled_np = scaler.transform(X_test)

In [20]:
X_train_standarized_np = scaler.transform(X_train)
X_test_standarized_np = scaler.transform(X_test)

X_train_standarized_df = pd.DataFrame(X_train_standarized_np, columns = X_train.columns, index=X_train.index)
X_test_standarized_df  = pd.DataFrame(X_test_standarized_np, columns = X_test.columns, index=X_test.index)

In [21]:
log_reg = LogisticRegression()
log_reg.fit(X_train_norm_df, y_train)
y_pred_test = log_reg.predict(X_test_norm_df)

In [25]:
print("Accuracy", log_reg.score(X_test_norm_df, y_test))
#print("F1 score", f1_score(y_test, pred))
print("Precision", precision_score(y_test, y_pred_test))
print("Recall", recall_score(y_test, y_pred_test)) ###double check

Accuracy 0.945495
Precision 0.9188630490956072
Recall 0.4090882945067587


In [34]:
from imblearn.over_sampling import RandomOverSampler

over = RandomOverSampler(random_state=0)
X_train_over, y_train_over = over.fit_resample(X_train_standarized_df,y_train)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_over, y_train_over)
y_pred_test_log = log_reg.predict(X_test_standarized_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



In [1]:
from imblearn.under_sampling import RandomUnderSampler

under = RandomUnderSampler(random_state=0)
X_train_under, y_train_under = under.fit_resample(X_train_standarized_df,y_train)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_under, y_train_under)
y_pred_test_log = log_reg.predict(X_test_standarized_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

NameError: name 'X_train_standarized_df' is not defined

In [29]:
from imblearn.over_sampling import SMOTE

In [32]:
sm = SMOTE(random_state = 1,sampling_strategy=1.0)
X_train_sm, y_train_sm =  sm.fit_resample(X_train_standarized_df,y_train)
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_sm, y_train_sm)
y_pred_test_log = log_reg.predict(X_test_standarized_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182615
         1.0       0.57      0.95      0.71     17385

    accuracy                           0.93    200000
   macro avg       0.78      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000

